# CMI with $S$

Calculating CMI with density matrices 

\begin{equation}

CMI = I(A:C|B) = S(AB) + S(BC) - S(ABC) - S(B),

\end{equation}

where $S(Q)$ is the von Neumann entropy of the $Q$ subsystem:

\begin{equation}

S(Q) = - tr ( \rho_{Q} \log \rho_Q ),

\end{equation}

with

\begin{equation}

\rho_{Q} = tr_{\bar{Q}} (\rho).

\end{equation}

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import math
from functools import reduce

We will start from defining the GHZ state:

\begin{equation}

\ket{\text{GHZ}}= \frac{1}{\sqrt{2}}(\ket{0}^{\otimes N}+ \ket{1}^{\otimes N})

\end{equation}

and the density matrix:

\begin{equation}

\rho_0 = \ket{\text{GHZ}} \bra{\text{GHZ}}

\end{equation}

In [34]:
#Defining rho_0

# -------------------------
# Tensor product helper
# -------------------------
def kron_power(v, N):
    """Compute v ⊗ v ⊗ ... ⊗ v (N times)"""
    return reduce(np.kron, [v] * N)

# -------------------------
# Parameters
# -------------------------
N = 12  # number of qubits

# -------------------------
# Basis states |0> and |1>
# -------------------------
ket_0 = np.array([1, 0])
ket_1 = np.array([0, 1])

# -------------------------
# Build product states
# |0...0> and |1...1>
# -------------------------
ket_0N = kron_power(ket_0, N)
ket_1N = kron_power(ket_1, N)

# -------------------------
# GHZ state
# -------------------------
ket_GHZ = (ket_0N + ket_1N) / np.sqrt(2)

print("GHZ state shape:", ket_GHZ.shape)

# -------------------------
# Density matrix
# -------------------------
rho_GHZ = np.outer(ket_GHZ, ket_GHZ.conj())
print(rho_GHZ)
print("Density matrix shape:", rho_GHZ.shape)


GHZ state shape: (4096,)
[[0.5 0.  0.  ... 0.  0.  0.5]
 [0.  0.  0.  ... 0.  0.  0. ]
 [0.  0.  0.  ... 0.  0.  0. ]
 ...
 [0.  0.  0.  ... 0.  0.  0. ]
 [0.  0.  0.  ... 0.  0.  0. ]
 [0.5 0.  0.  ... 0.  0.  0.5]]
Density matrix shape: (4096, 4096)


Subsequantly, we define the noise channel:

\begin{equation}

\mathcal{E}^i_p(\cdot) = (1-p)(\cdot) + pX_i(\cdot)X_i

\end{equation}

acting on every qubit.

In [35]:
#Defining the noise channel

#defining Pauli X matrix 

Pauli_X= np.array([[0, 1],[1, 0]])
print(Pauli_X)

#defining identity matrix 

I = np.eye(2)


#Defining local operator
def local_operator(op, i, N):
    """
    Build operator acting on i-th qubit (0-based index) in N-qubit system.
    """
    ops = [I] * N
    ops[i] = op
    return reduce(np.kron, ops)

#Defining the noise channel
def bit_flip_channel(rho, p, i, N):
    """ 
    Function implementing the application of the noise channel on qubit i. 
    Returns the matrix rho_p_i after the application of the noise on one qubit
    i - qubit number
    p - noise rate 
    N - total system size
    rho - total density matrix
    """
    X_i = local_operator(Pauli_X, i, N)
    return (1 - p) * rho + p * (X_i @ rho @ X_i)



[[0 1]
 [1 0]]


In [37]:
#Test 

# apply channel on qubit 0
p_test = 0.1
rho_out = bit_flip_channel(rho_GHZ, p_test, i=0, N=N)

print(rho_out)

[[0.45 0.   0.   ... 0.   0.   0.45]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.  ]
 ...
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.45 0.   0.   ... 0.   0.   0.45]]


In [ ]:
#Repeated application of bit flip chanell

def noise_channel(rho_0, p, N):
    """ 
    Function implementing the noise channel on qubit all qubits. 
    Returns the matrix rho_p_i after the application of the noise on one qubit
    i - qubit number
    p - noise rate 
    N - total system size
    rho_0 - initial density matrix
    """
    rho = rho_0
    for i in range(len(N)):
        rho = bit_flip_channel(rho, p, i, N)
    return rho 

In [ ]:
#test 

rho_out_2 = noise_channel(rho_GHZ, p_test, N=N)
print(rho_out_2)

Now, we will define functions to caluclate von Neuman entropies of the whole matrix and the reduced density matrices to subsequently calculate CMI. 

In [40]:
#Calculating von Neumann entropy

def von_neumann_entropy(rho, base=2):
    """
    Compute von Neumann entropy S(rho) = -Tr(rho log rho)
    base = np.e gives natural log, base=2 gives bits
    """
    # eigenvalues of rho
    eigvals = np.linalg.eigvalsh(rho)

    # keep only positive eigenvalues (avoid log(0))
    eigvals = eigvals[eigvals > 1e-12]

    S = -np.sum(eigvals * np.log(eigvals))

    if base == 2:
        S = S / np.log(2)

    return S

In [41]:
#Test

S = von_neumann_entropy(rho_out)
print("Von Neumann entropy:", S)

Von Neumann entropy: 0.46899559358928145


In [60]:
#Function to compute reduced density matrix


def partial_trace(rho, keep, N):
    """
    Partial trace over all subsystems NOT in 'keep'.

    Parameters:
    - rho: full density matrix (2^N x 2^N)
    - keep: list of subsystem indices to keep (e.g. [0,2])
    - N: number of qubits

    Returns:
    - reduced density matrix
    """
    dims = [2] * N
    # reshape into tensor with 2N indices
    rho = rho.reshape(dims + dims)

    # subsystems to trace out
    trace_out = sorted(set(range(N)) - set(keep), reverse=True)

    # trace out each unwanted subsystem
    for i in trace_out:
        rho = np.trace(rho, axis1=i, axis2=i + N)

    return rho

In [67]:
#Function to compute reduced density matrix

def partial_trace(rho, keep, N):
    """
    Partial trace over all subsystems NOT in 'keep'.

    Parameters:
    - rho: full density matrix (2^N x 2^N)
    - keep: list of subsystem indices to keep (e.g. [0,2])
    - N: number of qubits

    Returns:
    - reduced density matrix
    """

    dims = [2] * N

    rho_t = rho.reshape(dims + dims)

    keep = sorted(keep)

    # indices for full system
    all_idx = list(range(N))

    trace_idx = [i for i in all_idx if i not in keep]

    # build einsum string dynamically
    import string

    letters = string.ascii_lowercase + string.ascii_uppercase

    # assign indices for bra and ket spaces
    inds = letters[:N] + letters[N:2*N]

    # build contraction
    keep_bra = [inds[i] for i in keep]
    keep_ket = [inds[i + N] for i in keep]

    einsum_str = ''.join(inds) + '->' + ''.join(keep_bra + keep_ket)

    return np.einsum(einsum_str, rho_t)

Partition into subregions A,B,and C

In [61]:
#partition into subregions A,B,C

A = [0,1,2,3]

B = [4,5,10,11]

C = [6,7,8,9]


In [ ]:
def CMI(p,rho_0, A, B, C, N):
    dims = [2] * N

    rho = 

    S_AB = von_neumann_entropy(partial_trace(rho, A + B, N), base=2)
    S_BC = von_neumann_entropy(partial_trace(rho, B + C, N), base=2)
    S_B  = von_neumann_entropy(partial_trace(rho, B, N), base=2)

    S_ABC = von_neumann_entropy(rho, base=2)

    return S_AB + S_BC - S_B - S_ABC

In [65]:
print(A, B, C)
print(rho_out.shape)
print(N)

[0, 1, 2, 3] [4, 5, 10, 11] [6, 7, 8, 9]
(4096, 4096)
12


In [70]:
#test

cmi_test = CMI(rho_out,A=A,B=B,C=C,N=N)

print(cmi_test)

2.468995593589281
